# A3 cross-session re-ID on Lee 2019 OpenBMI motor imagery (54 subjects)

Replicates the BCI IV-2a A3 cross-session result on a 6× larger cohort. Lee 2019 has 54 subjects × 2 sessions on different days; binary left/right hand imagery. Pipeline mirrors `experiments/05`: victim trained on session-1 task labels, re-ID probe trained on session-1 embeddings, tested on session-2 embeddings.

Reads from the compact float16 cache produced by `A3_lee2019_download.ipynb`. **You must run that notebook first** so that `/content/drive/MyDrive/bci_cache/lee2019_mi/windowed/` contains all 108 .npz files before launching this one.

Chance top-1 = 1/54 ≈ 1.85% (vs IV-2a's 1/9 ≈ 11.1%). If A3 generalizes here, the cross-session claim is no longer anecdotal.

**Runtime → L4 GPU.** Expected wall: ~45-60 min (loading + training EEGNet from cache).

Send back the JSON between the BEGIN/END markers in the final cell.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Mount Drive cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.environ['BCI_LEE2019_CACHE'] = '/content/drive/MyDrive/bci_cache'
print('cache root:', os.environ['BCI_LEE2019_CACHE'])

## 2. Confirm cache is complete (must be 108/108 before continuing)

In [ ]:
import glob, sys
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Cached subject-sessions: {len(files)}/108')
missing = [i for i in range(1, 55)
    if not (os.path.isfile(f'{WIN}/subj{i:02d}_sess1.npz')
         and os.path.isfile(f'{WIN}/subj{i:02d}_sess2.npz'))]
if missing:
    sys.exit(f'!!! cache incomplete (missing subjects: {missing}); run A3_lee2019_download.ipynb first.')
print('cache complete — ready to run')

## 3. Clone repo + confirm GPU

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 4. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

## 5. Run experiment (all 54 subjects, 3 victims)

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.20_a3_lee2019 --all

## 6. Emit run metadata + paste-back result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = 'experiments.20_a3_lee2019'
ARGS = ['--all']
RESULTS = 'results/20_a3_lee2019.json'
EXTRAS = ['figures/20_a3_lee2019.pdf']
TAG = 'a3_lee2019'
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception:
    git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS] + EXTRAS}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — cell 5 did not finish. Re-run cell 5, then this cell.')
print('--- BEGIN 20_a3_lee2019.json ---')
print(open(RESULTS).read())
print('--- END 20_a3_lee2019.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')

## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/20_a3_lee2019.json')
files.download('figures/20_a3_lee2019.pdf')
files.download(f'runs/{run_id}/meta.json')